# 패스트볼 무브먼트 계층적 군집분석 (Hierarchical Clustering)

**변수**: 체공시간 보정 수직 무브먼트(`ivb_ft`), 체공시간 보정 수평 무브먼트(`hb_ft`), 팔 각도(`arm_angle`)
**방법**: 패스트볼(FF/SI/FC)을 모두 가져와 **구종 이름표 없이** 세 변수로 Ward 군집 → 끝난 뒤 원래 이름표와 비교

**필요 데이터**: Google Drive 원본 Statcast CSV → `data/raw/연도/statcast_YYYY-MM.csv`
이 노트북은 `notebooks/` 폴더에서 실행합니다.

**계산 코드는 `src/` 모듈에 있습니다.**
| 모듈 | 내용 |
|---|---|
| `src/utils/config.py` | 데이터 경로, 상수 |
| `src/preprocessing/clean_data.py` | CSV 불러오기, 품질 필터(결측 제거) |
| `src/preprocessing/feature_engineering.py` | 체공시간 계산, 보정 무브먼트 (가이드 1~9단계) |
| `src/models/hierarchical_clustering.py` | 군집 점 만들기, 표준화+Ward, 실루엣, 결과 요약 |
| `src/visualization/cluster_plots.py` | 덴드로그램, 산점도 |

In [ ]:
import os
import sys

# 현재 위치(notebooks)의 상위 폴더(프로젝트 최상단)를 경로에 추가 → src 모듈 import
sys.path.append(os.path.abspath(".."))

import pandas as pd

from src.utils.config import CLUSTER_FEATURES as FEATURES, PROCESSED_DIR, RAW_DIR
from src.preprocessing.clean_data import load_statcast, clean_missing_values
from src.preprocessing.feature_engineering import add_flight_adjusted_movement, movement_sanity_check
from src.models.hierarchical_clustering import (
    build_cluster_points, ward_linkage, silhouette_by_k, assign_clusters,
    summarize_clusters, label_crosstab, find_label_mismatches,
)
from src.visualization.cluster_plots import plot_dendrogram, plot_clusters, plot_by_label

# -----------------------------
# 설정값 (필요하면 여기만 바꾸세요)
# -----------------------------
PITCH_TYPES = ["FF", "SI", "FC"]
TRAIN_YEARS = [2021, 2022, 2023, 2024, 2025]    # 구종별 기준 체공시간(T_ref) 계산 연도
CLUSTER_YEARS = [2021, 2022, 2023, 2024, 2025]  # 군집분석에 포함할 시즌
UNIT = "pitcher_season"                         # "pitcher_season" 또는 "pitch"
MIN_PITCHES = 50                                # (pitcher_season) 투수-시즌-구종당 최소 투구 수
SAMPLE_SIZE = 8000                              # (pitch) 무작위 추출 투구 수
RANDOM_STATE = 42
N_CLUSTERS = 3                                  # 최종 군집 수

## 1. 데이터 불러오기 + 품질 필터
필요한 칼럼만 읽고, 정규시즌·피치아웃 제외·결측 투구 제거.

In [ ]:
data = load_statcast(RAW_DIR, pitch_types=PITCH_TYPES)
data = clean_missing_values(data)
data.groupby("game_year").size()

## 2. 체공시간 보정 무브먼트
`무브먼트 × (T_ref / T)²` — 자세한 식은 `src/preprocessing/feature_engineering.py` 참고.

In [ ]:
data, reference_times = add_flight_adjusted_movement(data, train_years=TRAIN_YEARS)
print("구종별 기준 체공시간(초)")
print(reference_times.round(4))

### 계산 점검 (가이드 23절)
체공시간 0.35~0.50초 비율 ≈ 1, 보정배율 ≈ 1, FF·SI의 `hb_ft` 양수(암사이드), FC는 음수 근처.

In [ ]:
movement_sanity_check(data)

## 3. 군집에 넣을 점 만들기
구종은 평균을 낼 때 공을 구분하는 데만 쓰고 **군집 변수에는 넣지 않습니다.**

In [ ]:
points = build_cluster_points(
    data, unit=UNIT, years=CLUSTER_YEARS,
    min_pitches=MIN_PITCHES, sample_size=SAMPLE_SIZE, random_state=RANDOM_STATE,
)
points["pitch_type"].value_counts()

## 4. 표준화 + Ward 계층적 군집

In [ ]:
X, Z = ward_linkage(points)
plot_dendrogram(Z);

## 5. 군집 수 고르기 (실루엣 점수)

In [ ]:
silhouette_by_k(X, Z).to_frame().T

## 6. 최종 군집 결과

In [ ]:
points = assign_clusters(points, Z, N_CLUSTERS)
summarize_clusters(points)

### 군집 vs 원래 구종 이름표 (각 군집 안 구종 비율 %)

In [ ]:
label_crosstab(points)

In [ ]:
plot_clusters(points);

In [ ]:
plot_by_label(points);

### 이름표와 실제 움직임이 다른 공
`pitch_type` = 원래 이름, `cluster_type` = 군집이 판단한 유형

In [ ]:
mismatch = find_label_mismatches(points)
print(f"이름표와 군집이 다른 공: {len(mismatch):,}개 / {len(points):,}개")

sort_col = "n_pitches" if "n_pitches" in mismatch.columns else "ivb_ft"
(
    mismatch.sort_values(sort_col, ascending=False)
    [["player_name", "game_year", "pitch_type", "cluster_type", sort_col] + FEATURES]
    .head(20)
    .round(2)
)

### 군집별 대표 예시

In [ ]:
sort_col = "n_pitches" if "n_pitches" in points.columns else "ivb_ft"
(
    points.sort_values(sort_col, ascending=False)
    .groupby("cluster").head(5)
    [["cluster", "player_name", "game_year", "pitch_type", sort_col] + FEATURES]
    .sort_values(["cluster", sort_col], ascending=[True, False])
    .round(2)
)

## 7. 결과 저장
`data/processed/`에 저장 (원본 `data/raw/`는 수정하지 않음).

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / f"hclust_fastballs_{UNIT}.csv"
points.to_csv(out_path, index=False)
print("저장 완료:", out_path.name)